## ***Read teh Dataset***

In [17]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total number of character: ", len(raw_text))

print(raw_text[: 99])

Total number of character:  20482
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


## **Step:1 Raw text to converting word and subword**

In [18]:
import re

text = "Hello, sir. This, is a test."

result = re.split(r'(\s)', text)

print(result)

['Hello,', ' ', 'sir.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


## **Removig also punctuation aslo**

In [19]:
result = re.split(r"([,.] | \s)", text)
result

['Hello', ', ', 'sir', '. ', 'This', ', ', 'is a test.']

## **Removing the white space**

In [20]:
result = [item for item in result if item.strip()]
result

['Hello', ', ', 'sir', '. ', 'This', ', ', 'is a test.']

In [21]:
text = "Hello, sir. This, is a test. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\'] |--|\s)', text)
result = [item for item in result if item.strip()]
print(result)

['Hello', ', ', 'sir', '. ', 'This', ', ', 'is', 'a', 'test', '. ', 'Is', 'this', '--', 'a', 'test?']


## **Now apply this is into our corpus**

In [22]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item for item in preprocessed if item.strip()]
print(preprocessed[: 20])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was']


In [23]:
len(preprocessed)

4690

## **Step2: Creating Token IDs**

In [24]:
## take all unique token
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
vocab_size

1130

In [25]:
print(all_words[:20])

['!', '"', "'", '(', ')', ',', '--', '.', ':', ';', '?', 'A', 'Ah', 'Among', 'And', 'Are', 'Arrt', 'As', 'At', 'Be']


In [26]:
datas = {
    "alamin": 1,
    "nsu": 2
}

print(datas.items())

dict_items([('alamin', 1), ('nsu', 2)])


In [27]:
word10 = all_words[40:50]
print(word10)

['Grafton', 'Greek', 'Grindle', 'Grindles', 'HAD', 'Had', 'Hang', 'Has', 'He', 'Her']


In [28]:
test_voc = {token:integer for integer, token in enumerate(word10)}
test_voc

{'Grafton': 0,
 'Greek': 1,
 'Grindle': 2,
 'Grindles': 3,
 'HAD': 4,
 'Had': 5,
 'Hang': 6,
 'Has': 7,
 'He': 8,
 'Her': 9}

In [29]:
# enumerate: assining all word under a integer.
vocab = {token:integer for integer, token in enumerate(all_words)}

## ***Assigning `token-id` in individuals token.***

In [32]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 30:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)


## **Custom tokenizer class**

In [33]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        ## replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)

        return text

## **Text into token id**

In [37]:
tokenizer = SimpleTokenizerV1(vocab)

text = """
It's the last he painted, you, Mrs. Gisburn said with pardonable pride.
"""

ids = tokenizer.encode(text)
print(ids)

[56, 2, 850, 988, 602, 533, 746, 5, 1126, 5, 67, 7, 38, 851, 1108, 754, 793, 7]


In [38]:
tokenizer.decode(ids)

"It' s the last he painted, you, Mrs. Gisburn said with pardonable pride."

## **If the token is not present in the vocal**

In [39]:
tokenizer.encode("hello")

KeyError: 'hello'

## **Adding special context tokens**

In [41]:
len(set(preprocessed))

1130

In [42]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token: integer for integer, token in enumerate(all_tokens)}

In [43]:
len(vocab.items())

1132

## **Last 5 form the `vocab`**

In [44]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [45]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]

        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        ## replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)

        return text

In [47]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [50]:
ids = tokenizer.encode(text)
ids

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [51]:
tokenizer.decode(ids)

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'